# Fetch - Usage Data.ipynb

## Overview

`Fetch - Usage Data.ipynb` is the **entry point** for the Sovereign Core Metrics Aggregator
notebook suite. It handles three critical tasks:

1. **Authentication** — Obtains a JWT bearer token from the Account IAM service
2. **Metrics Discovery** — Learns which metrics are available and how they are measured
3. **Data Fetching** — Retrieves raw, aggregated, and grouped usage data and writes it to `data/`

All downstream notebooks read from the files this notebook writes.
You do not need live API access to run them — the `data/` folder persists between sessions.

---

### What this notebook produces

| # | Section | Action | Output |
|---|---|---|---|
| 4 | Discover Available Metrics | `GET /metering/services/{serviceId}/metrics` | Prints available metric IDs and metering models |
| 5 | Fetch Raw Usage | `GET /metering/services/{serviceId}/usage/raw` | `data/raw/<domain>/<serviceId>/*.json` |
| 6 | Fetch Aggregated Usage | `GET /metering/services/{serviceId}/usage/aggregated` | `data/aggregated/<domain>/<serviceId>/*.json` |
| 7 | Fetch Grouped Usage | `GET /metering/services/{serviceId}/usage/aggregated/{groupBy}` | `data/grouped/<domain>/<serviceId>/*.json` |

---

### Prerequisites

- Copy `.env.template` → `.env` and fill in your `APP_DOMAIN`, `SERVICE_ID`, and `API_KEY`.
- Or run as-is — `.env.template` is loaded as fallback and the demo environment will be used.
- The notebook suppresses SSL warnings automatically (clusters may use self-signed certificates).

Once complete, run the downstream notebooks in any order:

| Notebook | Reads from |
|---|---|
| `Processing - Raw Usage Data.ipynb` | `data/raw/` |
| `Processing - Aggregated Usage Data.ipynb` | `data/aggregated/` |
| `Processing - Grouped Usage Data.ipynb` | `data/grouped/` |
| `Use Case - Telemetry.ipynb` | `data/aggregated/` + `data/grouped/` |
| `Use Case - Billing.ipynb` | `data/grouped/` |


## 1. Imports

Standard Python libraries for HTTP requests, JSON parsing, file I/O, and environment configuration.

In [7]:
import os
import json
import time
import datetime
import pathlib
import warnings
import requests
import urllib3
from dotenv import load_dotenv


## 2. Environment Variables & Configuration

### What This Section Does

1. **Loads credentials** from `.env` (or `.env.template` as fallback)
2. **Extracts configuration** into Python variables
3. **Constructs API URLs** based on your cluster domain
4. **Prints summary** (non-sensitive values only)

### Configuration Variables Explained

| Variable | Purpose | Example |
|---|---|---|
| `APP_DOMAIN` | Cluster domain where your Sovereign Core is deployed | `apps.example.cp.fyre.ibm.com` |
| `SERVICE_ID` | Catalog service ID you want to query | `cluster-as-a-service` |
| `TENANT_ID` | Tenant UUID for scoped queries, or `"all"` for cross-tenant | `"all"` |
| `IAM_ACCOUNT_TYPE` | Which account context to authenticate as | `"platform"` (MSP) |
| `API_KEY` | Credentials for the chosen IAM account | *(Not disclosed)* |
| `RAW_DATA_PATH` | Override for raw data directory (optional) | `<filepath>` |
| `AGGREGATED_DATA_PATH` | Override for aggregated data directory (optional) | `<filepath>` |
| `GROUPED_DATA_PATH` | Override for grouped data directory (optional) | `<filepath>` |

### Security Note

Never commit `.env` with real API keys. The `.env.template` file is safe — it contains placeholders only.

---

In [8]:
# Load .env if present, otherwise fall back to .env.template
if pathlib.Path(".env").exists():
    load_dotenv(".env")
    print("✅ Loaded from .env")
else:
    load_dotenv(".env.template")
    print("✅ Loaded from .env.template")

# Suppress SSL warnings — clusters may use self-signed certificates
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Extract configuration
APP_DOMAIN              = os.environ["APP_DOMAIN"]
SERVICE_ID              = os.environ["SERVICE_ID"]
TENANT_ID               = os.environ.get("TENANT_ID", "all")
IAM_ACCOUNT_TYPE        = os.environ.get("IAM_ACCOUNT_TYPE", "platform")
API_KEY                 = os.environ["API_KEY"]
RAW_DATA_PATH           = os.environ["RAW_DATA_PATH"]
AGGREGATED_DATA_PATH    = os.environ["AGGREGATED_DATA_PATH"]
GROUPED_DATA_PATH       = os.environ["GROUPED_DATA_PATH"]

# Derived URLs
BASE_URL     = f"https://sovereign-core-metrics-aggregator.{APP_DOMAIN}"
IAM_BASE_URL = f"https://account-iam.{APP_DOMAIN}"

# Display configuration (secrets never printed)
print(f"\n📋 Configuration Summary:")
print(f"  APP_DOMAIN            : {APP_DOMAIN}")
print(f"  SERVICE_ID            : {SERVICE_ID}")
print(f"  TENANT_ID             : {TENANT_ID}")
print(f"  IAM_ACCOUNT_TYPE      : {IAM_ACCOUNT_TYPE}")
print(f"  BASE_URL              : {BASE_URL}")
print(f"  IAM_BASE_URL          : {IAM_BASE_URL}")
print(f"  RAW_DATA_PATH         : {RAW_DATA_PATH}")
print(f"  AGGREGATED_DATA_PATH  : {AGGREGATED_DATA_PATH}")
print(f"  GROUPED_DATA_PATH     : {GROUPED_DATA_PATH}")

✅ Loaded from .env.template

📋 Configuration Summary:
  APP_DOMAIN            : apps.gori-agent-hub.cp.fyre.ibm.com
  SERVICE_ID            : servicebrokercore
  TENANT_ID             : platform
  IAM_ACCOUNT_TYPE      : platform
  BASE_URL              : https://sovereign-core-metrics-aggregator.apps.gori-agent-hub.cp.fyre.ibm.com
  IAM_BASE_URL          : https://account-iam.apps.gori-agent-hub.cp.fyre.ibm.com
  RAW_DATA_PATH         : 
  AGGREGATED_DATA_PATH  : 
  GROUPED_DATA_PATH     : 


## 3. Authentication

### How Authentication Works

This section obtains a **JWT bearer token** from the Account IAM service using your API key. All subsequent API calls use this token.

**Token Exchange Flow:**

```
1. Load API_KEY from environment
2. POST to Account IAM: /api/2.0/accounts/{IAM_ACCOUNT_TYPE}/apikeys/token
3. Receive JWT bearer token in response
4. Use token in Authorization header for all metrics API calls
```

### IAM Account Types

| Account Type | Use Case | Role |
|---|---|---|
| `platform` | MSP / Platform Admin | Billing Manager, can see all tenants |
| `global_account` | Internal Control Plane | System administrator role |
| `<tenant-uuid>` | Application Developer | Limited to specific tenant |

---

In [9]:
def get_bearer_token(iam_base_url: str, account_type: str, api_key: str) -> str:
    """Exchange an API key for a Bearer token via Account IAM."""
    url = f"{iam_base_url}/api/2.0/accounts/{account_type}/apikeys/token"
    response = requests.post(url, json={"apikey": api_key}, verify=False)
    response.raise_for_status()
    return response.json()["token"]


TOKEN = get_bearer_token(IAM_BASE_URL, IAM_ACCOUNT_TYPE, API_KEY)
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
print("✅ Token acquired successfully.")

✅ Token acquired successfully.


## 4. Discover Available Metrics

### Purpose

The Metrics Aggregator does **not hardcode** metric IDs. Instead:

1. Upstream metering systems emit `MeteredUsage` events with arbitrary `metricId` strings
2. The service normalises these events into an `InstanceMetric` table
3. The `/metrics` endpoint queries what's actually available
4. **Your valid metric IDs are discovered at runtime**

Each metric is paired with a **metering model** (`point-in-time`, `total-up-to-date`, or `high-watermark`) that describes how the quantity is reported. The metering model is critical when choosing the right aggregation transform — see Section 6 for details.

---

In [10]:
def fetch_metrics(base_url: str, service_id: str, headers: dict) -> list:
    """Discover available metrics for a service."""
    url = f"{base_url}/metering/services/{service_id}/metrics"
    response = requests.get(url, headers=headers, verify=False)
    response.raise_for_status()
    return response.json().get("metrics", [])


metrics = fetch_metrics(BASE_URL, SERVICE_ID, HEADERS)

# ── Execution summary ─────────────────────────────────────────────────
print(f"── Request ──────────────────────────────────────────────────")
print(f"GET {BASE_URL}/metering/services/{SERVICE_ID}/metrics")
print(f"")
print(f"── Response ─────────────────────────────────────────────────")
print(f"Metrics found: {len(metrics)}")
print(f"")
print(f"── Metrics ──────────────────────────────────────────────────")
for metric in metrics:
    metric_id = metric["metricId"]
    model = metric["meteringModel"]
    print(f"  {metric_id:<20} → {model}")

# Extract metric IDs for use in later examples
available_metric_ids = [m["metricId"] for m in metrics]


── Request ──────────────────────────────────────────────────
GET https://sovereign-core-metrics-aggregator.apps.gori-agent-hub.cp.fyre.ibm.com/metering/services/servicebrokercore/metrics

── Response ─────────────────────────────────────────────────
Metrics found: 3

── Metrics ──────────────────────────────────────────────────
  instances            → point-in-time
  users                → point-in-time
  api_calls            → total-up-to-date


## 5. Fetch Raw Usage

### Purpose

Raw usage provides **individual, time-stamped usage records** — one row per `(instance, metric, time-window)` pair.

**Use cases:**
- **Audits & Compliance:** Verify all usage events are recorded
- **Debugging:** Trace a specific instance's usage
- **Data Export:** Paginate through full dataset for reconciliation
- **Transaction Linking:** Use `correlationId` to trace back to upstream events

### Path Parameters

| Parameter | Type | Required | Description |
|---|---|---|---|
| `serviceId` | string | **Yes** | ID of a catalog service/product. Example: `cluster-as-a-service` |

### Query Parameters

| Parameter | Type | Required | Default | Description |
|---|---|---|---|---|
| `usageStart` | integer | **Yes** | — | UTC timestamp in milliseconds for start of usage period. Must be > 1746403200000 (May 5 2026 00:00 UTC). |
| `usageEnd` | integer | No | Current time | UTC timestamp in milliseconds for end of usage period. Must be > `usageStart`. |
| `tenantIds` | string | No | `"all"` | Comma-separated tenant IDs. Use `"all"` for all tenants. API key will limit scope based on tenant permissions. |
| `workspaceIds` | string | No | `"all"` | Comma-separated workspace IDs. Use `"all"` for all workspaces. |
| `instanceIds` | string | No | `"all"` | Comma-separated instance IDs. Use `"all"` for all instances. |
| `metricIds` | string | No | `"all"` | Comma-separated metric IDs. Use `"all"` for all metrics. Can specify multiple values in one call. |
| `usagePageSize` | integer | No | 100 | Maximum records per page. Range: 1–10,000. |
| `usageNextPageKey` | integer | No | 0 | Usage ID to fetch the next page. Pass the value of the last `id` from the previous response to continue pagination. |

### Sample Response

```json
{
  "params": {
    "tenantIds": ["all"],
    "workspaceIds": ["all"],
    "instanceIds": ["all"],
    "metricIds": ["all"],
    "usageStart": 1781782340254,
    "usageEnd": 1791792340254,
    "usagePageSize": 10000,
    "usageNextPageKey": 0,
    "serviceId": "cluster-as-a-service"
  },
  "meteredUsage": [
    {
      "instanceId": "019f9373-66fb-70e9-bac2-93ee901f3d06",
      "tenantId": "platform",
      "workspaceId": "asaservice",
      "serviceId": "cluster-as-a-service",
      "region": "earth-1",
      "crn": "crn:v1:ibm-sc:private:cluster-as-a-service:earth-1:sub/asaservice:019f9373-66fb-70e9-bac2-93ee901f3d06::",
      "metricId": "cluster",
      "meteringModel": "point-in-time",
      "id": 1976,
      "startTimestamp": "2026-07-24T10:41:08.586Z",
      "endTimestamp": "2026-07-24T10:41:08.587Z",
      "usageQuantity": 1,
      "correlationId": "test-storage-1-provisioned-2026-07-24T10:41:08Z",
      "transactionId": "cluster-as-a-service-1c0a0707-0a01-4857-8248-2879078663b7-1784889669295"
    }
  ]
}
```

### Response Fields

**Top-level:**

| Field | Type | Description |
|---|---|---|
| `params` | object | Echo of the query parameters used for this request. |
| `meteredUsage` | array | List of raw usage records matching the query filters. |

**Per record (`meteredUsage[]`):**

| Field | Type | Description |
|---|---|---|
| `id` | integer | Record ID — used for pagination via `usageNextPageKey`. |
| `startTimestamp` | string | ISO 8601 timestamp with timezone. Start of the usage measurement period. |
| `endTimestamp` | string | ISO 8601 timestamp with timezone. End of the usage measurement period. |
| `crn` | string | Cloud Resource Name — globally unique resource identifier. Format: `crn:v1:{cName}:{ctype}:{serviceId}:{region}:sub/{workspaceId}:{instanceId}::` |
| `instanceId` | string | Unique identifier for the service instance. Extracted from `crn`. |
| `tenantId` | string | ID of the tenant that owns this instance. |
| `workspaceId` | string | ID of the workspace this instance belongs to. Extracted from `crn`. |
| `serviceId` | string | ID of the catalog service this instance is running. Extracted from `crn`. |
| `region` | string | Region where this instance is deployed (e.g., `us-east-1`). Extracted from `crn`. |
| `metricId` | string | ID of the metric being reported (e.g., `cluster`, `cpu_cores`, `memory_gb`). |
| `meteringModel` | string | Metering model for this usage event: `point-in-time`, `total-up-to-date`, or `high-watermark`. |
| `usageQuantity` | number | The measured value for this metric during the period. Unit is metric-specific. |
| `correlationId` | string | Correlation ID for tracing the submission across services. |
| `transactionId` | string | Unique identifier for this usage submission. |

---

In [11]:
def fetch_raw_usage(base_url: str, service_id: str, headers: dict, params: dict) -> dict:
    """Fetch one page of raw usage data."""
    url = f"{base_url}/metering/services/{service_id}/usage/raw"
    response = requests.get(url, headers=headers, params=params, verify=False)
    response.raise_for_status()
    return response.json()


def save_json(data: dict, path: str) -> None:
    pathlib.Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)


now_ms    = int(time.time() * 1000)
day_ms    = 24 * 60 * 60 * 1000
week_ms   = 7 * day_ms
month_ms  = 30 * day_ms

raw_examples = [
    {
        "label": "last_24h_all_tenants",
        "params": {
            "tenantIds": "all",
            "workspaceIds": "all",
            "instanceIds": "all",
            "usageStart": now_ms - day_ms,
            "usageEnd": now_ms,
            "usagePageSize": 10000,
        },
    },
    {
        "label": "last_7d_all_tenants",
        "params": {
            "tenantIds": "all",
            "workspaceIds": "all",
            "instanceIds": "all",
            "usageStart": now_ms - week_ms,
            "usageEnd": now_ms,
            "usagePageSize": 10000,
        },
    },
    {
        "label": "last_30d_specific_tenant",
        "params": {
            "tenantIds": TENANT_ID,
            "workspaceIds": "all",
            "instanceIds": "all",
            "usageStart": now_ms - month_ms,
            "usageEnd": now_ms,
            "usagePageSize": 10000,
        },
    },
    {
        "label": "last_30d_all_tenant",
        "params": {
            "tenantIds": "all",
            "workspaceIds": "all",
            "instanceIds": "all",
            "usageStart": now_ms - month_ms,
            "usageEnd": now_ms,
            "usagePageSize": 10000,
        },
    },
    {
        "label": "last_24h_specific_tenant_paginated",
        "params": {
            "tenantIds": TENANT_ID,
            "workspaceIds": "all",
            "instanceIds": "all",
            "usageStart": now_ms - day_ms,
            "usageEnd": now_ms,
            "usagePageSize": 50,
        },
    },
]

for example in raw_examples:
    p = example["params"]
    out_path = f"data/raw/{APP_DOMAIN}/{SERVICE_ID}/{example['label']}.json"
    data = fetch_raw_usage(BASE_URL, SERVICE_ID, HEADERS, p)
    save_json(data, out_path)
    records   = data.get("meteredUsage", [])
    has_more  = data.get("hasMore", False)
    start_dt  = datetime.datetime.utcfromtimestamp(p['usageStart'] / 1000).strftime('%Y-%m-%d')
    end_dt    = datetime.datetime.utcfromtimestamp(p['usageEnd']   / 1000).strftime('%Y-%m-%d')
    print(f"── {example['label']} " + "─" * max(0, 52 - len(example['label'])))
    print(f"   Request : GET .../usage/raw")
    print(f"   Params  : tenantIds={p['tenantIds']}  "
          f"usageStart={start_dt}  usageEnd={end_dt}  pageSize={p['usagePageSize']}")
    print(f"   Response: {len(records)} records  hasMore={has_more}")
    print(f"   Saved   : {out_path}")
    if records:
        sample = {k: v for k, v in list(records[0].items())[:6]}
        print(f"   Sample  : {json.dumps(sample)}")
    print()


/var/folders/mh/rhl3j_v5075fwz2dzshygytr0000gn/T/ipykernel_77376/644437761.py:85: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  start_dt  = datetime.datetime.utcfromtimestamp(p['usageStart'] / 1000).strftime('%Y-%m-%d')
/var/folders/mh/rhl3j_v5075fwz2dzshygytr0000gn/T/ipykernel_77376/644437761.py:86: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  end_dt    = datetime.datetime.utcfromtimestamp(p['usageEnd']   / 1000).strftime('%Y-%m-%d')


── last_24h_all_tenants ────────────────────────────────
   Request : GET .../usage/raw
   Params  : tenantIds=all  usageStart=2026-08-23  usageEnd=2026-08-24  pageSize=10000
   Response: 10000 records  hasMore=False
   Saved   : data/raw/apps.gori-agent-hub.cp.fyre.ibm.com/servicebrokercore/last_24h_all_tenants.json
   Sample  : {"instanceId": "019ff75c-0850-71fc-91d1-b7f999237a7b", "tenantId": "0fd2a329-b354-44ad-90e7-16f00fa85948", "workspaceId": "20260812-1434-3899-16f9-16230953af1d", "serviceId": "servicebrokercore", "region": "earth-1", "crn": "crn:v1:ibm-sc:private:servicebrokercore:earth-1:sub/20260812-1434-3899-16f9-16230953af1d:019ff75c-0850-71fc-91d1-b7f999237a7b::"}

── last_7d_all_tenants ─────────────────────────────────
   Request : GET .../usage/raw
   Params  : tenantIds=all  usageStart=2026-08-17  usageEnd=2026-08-24  pageSize=10000
   Response: 10000 records  hasMore=False
   Saved   : data/raw/apps.gori-agent-hub.cp.fyre.ibm.com/servicebrokercore/last_7d_all_tenants

## 6. Fetch Aggregated Usage

### Purpose

Retrieves usage data grouped into time periods for charting and billing purposes. The caller selects the transform (`sum`, `avg`, `min`, `max`) appropriate to their needs — the API does not restrict which transform can be used with a given metering model. Note that usage data can be amended up to 24 hours in the past, so callers should account for that window when determining the query period.

**Use cases:**
- **Billing Summaries:** Monthly total requests or resource consumption
- **Dashboards:** Daily or hourly trends of CPU, memory, users
- **Anomaly Detection:** Find spikes or drops in usage
- **Capacity Planning:** Understand peak usage patterns over time

### Metering Models & Transform Selection

Each metric has a **metering model** that determines which transform to use:

| Model | Meaning | Example | Recommended Transform |
|---|---|---|---|
| `point-in-time` | Latest/most recent value for the time period | CPU utilization, current memory | `avg` (typical), `max` (peak), `min` (floor) |
| `total-up-to-date` | Sum of all submissions in the time period | Total users, total API calls | `sum` (total across periods) |
| `high-watermark` | Maximum observed value; tracks the peak | Peak memory used, max concurrent connections | `avg` (the mean value) |

 **Note**
 * on `point-in-time` metrics: Do NOT use `sum` on point-in-time snapshots—summing latest values across periods has no semantic meaning. Use `avg` (typical), `max` (peak), or `min` (floor) instead.
 
 * Using `sum` on `total-up-to-date` metrics gives you the total accumulated usage
 
 * Always check the metering model available for the service in Section 4 before choosing a transform.

### Transform Options

| Transform | Description |
|---|---|
| `max` | Largest value in the period |
| `avg` | Mean of all values in the period |
| `min` | Smallest value in the period | 
| `sum` | Total of all values in the period | 

### Path Parameters

| Parameter | Type | Required | Description |
|---|---|---|---|
| `serviceId` | string | **Yes** | ID of a catalog service/product. Example: `cluster-as-a-service` |

### Query Parameters

| Parameter | Type | Required | Default | Description |
|---|---|---|---|---|
| `metricId` | string | **Yes** | — | The name of the metric to aggregate. One metric per call. |
| `transform` | string | **Yes** | — | Aggregation operation applied per period: `sum`, `avg`, `min`, or `max`. |
| `usageStart` | integer | **Yes** | — | UTC timestamp in milliseconds for the start of the usage period. |
| `usageEnd` | integer | No | Current time | UTC timestamp in milliseconds for the end of the usage period. |
| `groupByHrs` | integer | No | Full range | Time period size in hours (1–8760). If omitted, all data collapses into a single period. |
| `tenantIds` | string | No | `"all"` | Comma-separated tenant IDs. Use `"all"` for all tenants. |
| `workspaceIds` | string | No | `"all"` | Comma-separated workspace IDs. Use `"all"` for all workspaces. |
| `instanceIds` | string | No | `"all"` | Comma-separated instance IDs. Use `"all"` for all instances. |

### Sample Response

```json
{
  "params": {
    "tenantIds": ["all"],
    "workspaceIds": ["all"],
    "instanceIds": ["all"],
    "usageStart": 1781782340254,
    "usageEnd": 1791792340254,
    "groupByHrs": 1,
    "transform": "sum",
    "metricId": "users",
    "serviceId": "cluster-as-a-service"
  },
  "totalPeriodsCount": 129,
  "aggregatedMeteredUsagePeriods": [
    {
      "periodNumber": 1,
      "periodQuantity": 4,
      "periodStart": 1785110400000,
      "periodEnd": 1785114000000
    },
    {
      "periodNumber": 2,
      "periodQuantity": 32,
      "periodStart": 1785236400000,
      "periodEnd": 1785240000000
    },
    {
      "periodNumber": 3,
      "periodQuantity": 20,
      "periodStart": 1785240000000,
      "periodEnd": 1785243600000
    },
    ...
    {
      "periodNumber": 129,
      "periodQuantity": 8,
      "periodStart": 1785888000000,
      "periodEnd": 1785891600000
    }
  ]
}
```

### Response Fields

**Top-level:**

| Field | Type | Description |
|---|---|---|
| `params` | object | Echo of the request parameters as interpreted by the server. |
| `totalPeriodsCount` | integer | Total number of period entries returned in `aggregatedMeteredUsagePeriods`. |
| `aggregatedMeteredUsagePeriods` | array | List of time-bucketed aggregation results, ordered chronologically. Only periods with at least one usage event are included — empty buckets are omitted. |

**Per period (`aggregatedMeteredUsagePeriods[]`):**

| Field | Type | Description |
|---|---|---|
| `periodNumber` | integer | Sequential index of this period (1-based). |
| `periodQuantity` | number | Result of the chosen `transform` applied to all usage values within this period. Unit is metric-specific. |
| `periodStart` | integer | UTC timestamp in milliseconds for the start of this period. |
| `periodEnd` | integer | UTC timestamp in milliseconds for the end of this period. |

In [12]:
def fetch_aggregated_usage(base_url: str, service_id: str, headers: dict, params: dict) -> dict:
    """Fetch one page of aggregated usage data."""
    url = f"{base_url}/metering/services/{service_id}/usage/aggregated"
    response = requests.get(url, headers=headers, params=params, verify=False)
    response.raise_for_status()
    return response.json()


# ── Transform selection rules (based on meteringModel) ────────────────────
#
# point-in-time    → avg (typical), max (peak), or min (floor) — NOT sum
# total-up-to-date → sum (total across periods)
# high-watermark   → avg (mean value)
# unknown model    → avg  (safe fallback with a printed warning)
#
MODEL_TRANSFORM = {
    "point-in-time":    "avg",
    "total-up-to-date": "sum",
    "high-watermark":   "avg",
}

# ── Bucket-size overrides ─────────────────────────────────────────────────
#
# Most metrics are fine at daily (24h) resolution.
# List any metric IDs here that benefit from a finer bucket size.
# Example: memory_gb at 1h gives the time-of-day heatmap in Processing - Aggregated Usage Data.ipynb.
#
BUCKET_OVERRIDES: dict[str, int] = {
    "memory_gb": 1,   # hourly → enables hour-of-day heatmap
}
DEFAULT_BUCKET_HRS = 24

# ── Build aggregated_examples from discovered metrics ─────────────────────
aggregated_examples = []
for m in metrics:
    metric_id = m["metricId"]
    model     = m["meteringModel"]
    transform = MODEL_TRANSFORM.get(model)

    if transform is None:
        print(f"  ⚠ Unknown meteringModel '{model}' for metric '{metric_id}' "
              f"— defaulting to transform='avg'")
        transform = "avg"

    bucket_hrs = BUCKET_OVERRIDES.get(metric_id, DEFAULT_BUCKET_HRS)
    window_tag = "last_30d" if bucket_hrs >= 24 else "last_7d"
    freq_tag   = "daily"   if bucket_hrs >= 24 else f"{bucket_hrs}h"
    label      = f"{window_tag}_{freq_tag}_{transform}_{metric_id}"

    aggregated_examples.append({
        "label":     label,
        "metric":    metric_id,
        "model":     model,
        "transform": transform,
        "groupByHrs": bucket_hrs,
    })

print(f"Built {len(aggregated_examples)} aggregated example(s) from discovered metrics:")
for ex in aggregated_examples:
    print(f"  {ex['metric']:<22} model={ex['model']:<20} "
          f"→ transform={ex['transform']:<4}  "
          f"bucket={ex['groupByHrs']}h  label={ex['label']}")

# ── Fetch and save ────────────────────────────────────────────────────────
print()
for example in aggregated_examples:
    window_ms = month_ms if example["groupByHrs"] >= 24 else week_ms
    p = {
        "tenantIds":    "all",
        "workspaceIds": "all",
        "instanceIds":  "all",
        "usageStart":   now_ms - window_ms,
        "usageEnd":     now_ms,
        "groupByHrs":   example["groupByHrs"],
        "transform":    example["transform"],
        "metricId":     example["metric"],
    }
    out_path = f"data/aggregated/{APP_DOMAIN}/{SERVICE_ID}/{example['label']}.json"
    data = fetch_aggregated_usage(BASE_URL, SERVICE_ID, HEADERS, p)
    save_json(data, out_path)
    periods       = data.get("aggregatedMeteredUsagePeriods", [])
    total_periods = data.get("totalPeriodsCount", 0)
    start_dt = datetime.datetime.utcfromtimestamp(p['usageStart'] / 1000).strftime('%Y-%m-%d')
    end_dt   = datetime.datetime.utcfromtimestamp(p['usageEnd']   / 1000).strftime('%Y-%m-%d')
    print(f"── {example['label']} " + "─" * max(0, 52 - len(example['label'])))
    print(f"   Request : GET .../usage/aggregated")
    print(f"   Params  : metricId={example['metric']}  transform={example['transform']}  "
          f"bucket={example['groupByHrs']}h  {start_dt} → {end_dt}")
    print(f"   Response: {len(periods)} periods  totalPeriodsCount={total_periods}")
    print(f"   Saved   : {out_path}")
    if periods:
        sample = {k: v for k, v in list(periods[0].items())[:5]}
        print(f"   Sample  : {json.dumps(sample)}")
    print()


Built 3 aggregated example(s) from discovered metrics:
  instances              model=point-in-time        → transform=avg   bucket=24h  label=last_30d_daily_avg_instances
  users                  model=point-in-time        → transform=avg   bucket=24h  label=last_30d_daily_avg_users
  api_calls              model=total-up-to-date     → transform=sum   bucket=24h  label=last_30d_daily_sum_api_calls



/var/folders/mh/rhl3j_v5075fwz2dzshygytr0000gn/T/ipykernel_77376/4282331631.py:83: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  start_dt = datetime.datetime.utcfromtimestamp(p['usageStart'] / 1000).strftime('%Y-%m-%d')
/var/folders/mh/rhl3j_v5075fwz2dzshygytr0000gn/T/ipykernel_77376/4282331631.py:84: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  end_dt   = datetime.datetime.utcfromtimestamp(p['usageEnd']   / 1000).strftime('%Y-%m-%d')


── last_30d_daily_avg_instances ────────────────────────
   Request : GET .../usage/aggregated
   Params  : metricId=instances  transform=avg  bucket=24h  2026-07-25 → 2026-08-24
   Response: 13 periods  totalPeriodsCount=30
   Saved   : data/aggregated/apps.gori-agent-hub.cp.fyre.ibm.com/servicebrokercore/last_30d_daily_avg_instances.json
   Sample  : {"periodNumber": 18, "periodQuantity": 1, "periodStart": 1786406400000, "periodEnd": 1786492800000}

── last_30d_daily_avg_users ────────────────────────────
   Request : GET .../usage/aggregated
   Params  : metricId=users  transform=avg  bucket=24h  2026-07-25 → 2026-08-24
   Response: 13 periods  totalPeriodsCount=30
   Saved   : data/aggregated/apps.gori-agent-hub.cp.fyre.ibm.com/servicebrokercore/last_30d_daily_avg_users.json
   Sample  : {"periodNumber": 18, "periodQuantity": 1, "periodStart": 1786406400000, "periodEnd": 1786492800000}

── last_30d_daily_sum_api_calls ────────────────────────
   Request : GET .../usage/aggregated
 

## 7. Fetch Aggregated Usage by Group

### Purpose

Retrieves aggregated usage data broken down by a grouping dimension and further grouped into time periods for charting and billing purposes. Unlike the base `/aggregated` endpoint — which collapses all matching data into a single list of time periods — this endpoint produces **one set of time periods per group member** (e.g., one set per tenant when `groupBy=groupByTenant`). This allows callers to compare usage across tenants, workspaces, or instances over the same time window.

The caller selects the transform appropriate to their needs — the API does not restrict which transform can be used with a given metering model. Note that usage data can be amended up to 24 hours in the past, so callers should account for that window when determining the query period.

**Use cases:**
- **Multi-tenant comparison:** Compare resource consumption across all tenants side by side
- **Workspace breakdown:** Understand which workspaces are driving usage within a tenant
- **Per-instance analysis:** Drill down to individual instance-level usage trends

### Path Parameters

| Parameter | Type | Required | Description |
|---|---|---|---|
| `serviceId` | string | **Yes** | ID of a catalog service/product. Example: `cluster-as-a-service` |
| `groupBy` | string | **Yes** | Grouping dimension for the response: `groupByTenant`, `groupByWorkspace`, or `groupByInstance`. This is a path parameter because it fundamentally changes the shape of the response — specifically the identity fields carried on each entry. |

### Query Parameters

| Parameter | Type | Required | Default | Description |
|---|---|---|---|---|
| `metricId` | string | **Yes** | — | The name of the metric to aggregate. One metric per call. |
| `transform` | string | **Yes** | — | Aggregation operation applied per period: `sum`, `avg`, `min`, or `max`. |
| `usageStart` | integer | **Yes** | — | UTC timestamp in milliseconds for start of usage period. |
| `usageEnd` | integer | No | Current time | UTC timestamp in milliseconds for end of usage period. |
| `groupByHrs` | integer | No | Full range | Time period size in hours (1–8760). If omitted, all data is grouped into a single period spanning `usageStart` to `usageEnd`. |
| `tenantIds` | string | No | `"all"` | Comma-separated tenant IDs. Use `"all"` for all tenants. API key will limit scope based on tenant permissions. |
| `workspaceIds` | string | No | `"all"` | Comma-separated workspace IDs. Use `"all"` for all workspaces. |
| `instanceIds` | string | No | `"all"` | Comma-separated instance IDs. Use `"all"` for all instances. |

### groupId Format

Each period entry carries a `groupId` identifying which group it belongs to. The format is always `<tenantId>:<workspaceId>:<instanceId>` with exactly two `:` separators. Segments not relevant to the active grouping dimension are left blank:

| `groupBy` value | `groupId` format | Example |
|---|---|---|
| `groupByTenant` | `<tenantId>::` | `de210e42-eb19-491b-a55e-58078e0b0d97::` |
| `groupByWorkspace` | `<tenantId>:<workspaceId>:` | `de210e42-eb19-491b-a55e-58078e0b0d97:20260623-1313-2638-6634-e079ed7eab42:` |
| `groupByInstance` | `<tenantId>:<workspaceId>:<instanceId>` | `de210e42-eb19-491b-a55e-58078e0b0d97:20260623-1313-2638-6634-e079ed7eab42:019ef4a6-6677-77be-96a6-53ece7a4ad36` |

Entries with the same `groupId` form a contiguous, chronologically ordered set of periods for that group. Each entry carries only the identity fields relevant to the active `groupBy` dimension — absent fields are omitted entirely. Every group always returns the same number of periods (`totalPeriodsPerGroup`); if a group had no usage in a bucket, `periodQuantity` is `0` rather than omitted, keeping all groups the same length.

### Sample Response — `groupByWorkspace`

```json
{
  "params": {
    "tenantIds": ["all"],
    "workspaceIds": ["all"],
    "instanceIds": ["all"],
    "usageStart": 1781782340254,
    "usageEnd": 1791792340254,
    "groupByHrs": 24,
    "transform": "avg",
    "metricId": "cpu_cores",
    "serviceId": "cluster-as-a-service",
    "groupBy": "groupByWorkspace"
  },
  "totalGroups": 1,
  "totalPeriodsPerGroup": 81,
  "totalPeriodsCount": 81,
  "aggregatedMeteredUsagePeriods": [
    {
      "groupId": "platform:asaservice:",
      "groupTenantId": "platform",
      "groupWorkspaceId": "asaservice",
      "periodNumber": 1,
      "periodQuantity": 6.117647058823529,
      "periodStart": 1784851200000,
      "periodEnd": 1784937600000
    },
    {
      "groupId": "platform:asaservice:",
      "groupTenantId": "platform",
      "groupWorkspaceId": "asaservice",
      "periodNumber": 2,
      "periodQuantity": 8,
      "periodStart": 1784937600000,
      "periodEnd": 1785024000000
    },
    {
      "groupId": "platform:asaservice:",
      "groupTenantId": "platform",
      "groupWorkspaceId": "asaservice",
      "periodNumber": 3,
      "periodQuantity": 8,
      "periodStart": 1785024000000,
      "periodEnd": 1785110400000
    },
    ...
  ]
}
```

### Sample Response — `groupByInstance`

```json
{
  "params": {
    "tenantIds": ["all"],
    "workspaceIds": ["all"],
    "instanceIds": ["all"],
    "usageStart": 1781782340254,
    "usageEnd": 1791792340254,
    "groupByHrs": 24,
    "transform": "avg",
    "metricId": "cpu_cores",
    "serviceId": "cluster-as-a-service",
    "groupBy": "groupByInstance"
  },
  "totalGroups": 1,
  "totalPeriodsPerGroup": 81,
  "totalPeriodsCount": 81,
  "aggregatedMeteredUsagePeriods": [
    {
      "groupId": "platform:asaservice:019f9373-66fb-70e9-bac2-93ee901f3d06",
      "groupTenantId": "platform",
      "groupWorkspaceId": "asaservice",
      "groupInstanceId": "019f9373-66fb-70e9-bac2-93ee901f3d06",
      "periodNumber": 1,
      "periodQuantity": 6.117647058823529,
      "periodStart": 1784851200000,
      "periodEnd": 1784937600000
    },
    {
      "groupId": "platform:asaservice:019f9373-66fb-70e9-bac2-93ee901f3d06",
      "groupTenantId": "platform",
      "groupWorkspaceId": "asaservice",
      "groupInstanceId": "019f9373-66fb-70e9-bac2-93ee901f3d06",
      "periodNumber": 2,
      "periodQuantity": 8,
      "periodStart": 1784937600000,
      "periodEnd": 1785024000000
    },
    {
      "groupId": "platform:asaservice:019f9373-66fb-70e9-bac2-93ee901f3d06",
      "groupTenantId": "platform",
      "groupWorkspaceId": "asaservice",
      "groupInstanceId": "019f9373-66fb-70e9-bac2-93ee901f3d06",
      "periodNumber": 3,
      "periodQuantity": 8,
      "periodStart": 1785024000000,
      "periodEnd": 1785110400000
    },
    ...
  ]
}
```

### Response Fields

**Top-level:**

| Field | Type | Description |
|---|---|---|
| `params` | object | Echo of the query parameters used for this request. |
| `totalGroups` | integer | Total number of distinct groups found (e.g., number of tenants, workspaces, or instances). |
| `totalPeriodsPerGroup` | integer | Number of time periods per group. Every group returns this many periods, including zero-quantity periods. |
| `totalPeriodsCount` | integer | Total number of period entries in the response, ie. sum across groups. |
| `aggregatedMeteredUsagePeriods` | array | Flat list of period entries ordered by group, then by `periodStart` ascending within each group. |

**Per period (`aggregatedMeteredUsagePeriods[]`):**

| Field | Type | Present when | Description |
|---|---|---|---|
| `groupId` | string | Always | Composite group identifier in the format `<tenantId>:<workspaceId>:<instanceId>`. Segments irrelevant to the active `groupBy` dimension are left blank. |
| `groupTenantId` | string | Always | Tenant ID of this group. |
| `groupWorkspaceId` | string | `groupByWorkspace`, `groupByInstance` | Workspace ID of this group. |
| `groupInstanceId` | string | `groupByInstance` only | Instance ID of this group. |
| `periodNumber` | integer | Always | Sequential period index within this group, starting at 1. |
| `periodQuantity` | number | Always | Result of the chosen `transform` applied to all usage values within this period. `0` if the group had no usage in this bucket. |
| `periodStart` | integer | Always | UTC timestamp in milliseconds for the start of this period. |
| `periodEnd` | integer | Always | UTC timestamp in milliseconds for the end of this period. |

---

In [13]:
def fetch_aggregated_usage_by_group(base_url: str, service_id: str, group_by: str, headers: dict, params: dict) -> dict:
    """Fetch aggregated usage data broken down by a grouping dimension."""
    url = f"{base_url}/metering/services/{service_id}/usage/aggregated/{group_by}"
    response = requests.get(url, headers=headers, params=params, verify=False)
    response.raise_for_status()
    return response.json()


# ── groupBy dimensions to fetch for every metric ─────────────────────────
#
# All three dimensions are always fetched — one file per (metric, dimension).
# This gives the grouped visualisation notebook full flexibility to show
# tenant-level, workspace-level, and instance-level breakdowns.
#
GROUP_DIMENSIONS = ["groupByTenant", "groupByWorkspace", "groupByInstance"]
GROUP_SUFFIX     = {
    "groupByTenant":    "by_tenant",
    "groupByWorkspace": "by_workspace",
    "groupByInstance":  "by_instance",
}

# ── Build group_examples from the same discovered metrics ─────────────────
# Reuses MODEL_TRANSFORM and BUCKET_OVERRIDES defined in Section 6.
#
group_examples = []
for m in metrics:
    metric_id = m["metricId"]
    model     = m["meteringModel"]
    transform = MODEL_TRANSFORM.get(model, "avg")
    bucket_hrs = BUCKET_OVERRIDES.get(metric_id, DEFAULT_BUCKET_HRS)
    window_tag = "last_30d" if bucket_hrs >= 24 else "last_7d"
    freq_tag   = "daily"   if bucket_hrs >= 24 else f"{bucket_hrs}h"

    for dim in GROUP_DIMENSIONS:
        suffix = GROUP_SUFFIX[dim]
        label  = f"{window_tag}_{freq_tag}_{transform}_{metric_id}_{suffix}"
        group_examples.append({
            "label":     label,
            "group_by":  dim,
            "metric":    metric_id,
            "model":     model,
            "transform": transform,
            "groupByHrs": bucket_hrs,
        })

print(f"Built {len(group_examples)} grouped example(s) "
      f"({len(metrics)} metric(s) × {len(GROUP_DIMENSIONS)} dimensions):")
for ex in group_examples:
    print(f"  {ex['metric']:<22} {ex['group_by']:<18} "
          f"→ transform={ex['transform']:<4}  label={ex['label']}")

# ── Fetch and save ────────────────────────────────────────────────────────
print()
for example in group_examples:
    window_ms = month_ms if example["groupByHrs"] >= 24 else week_ms
    p = {
        "tenantIds":    "all",
        "workspaceIds": "all",
        "instanceIds":  "all",
        "usageStart":   now_ms - window_ms,
        "usageEnd":     now_ms,
        "groupByHrs":   example["groupByHrs"],
        "transform":    example["transform"],
        "metricId":     example["metric"],
    }
    out_path = f"data/grouped/{APP_DOMAIN}/{SERVICE_ID}/{example['label']}.json"
    data = fetch_aggregated_usage_by_group(
        BASE_URL, SERVICE_ID, example["group_by"], HEADERS, p
    )
    save_json(data, out_path)
    total_groups  = data.get("totalGroups", 0)
    total_periods = data.get("totalPeriodsCount", 0)
    periods       = data.get("aggregatedMeteredUsagePeriods", [])
    start_dt = datetime.datetime.utcfromtimestamp(p['usageStart'] / 1000).strftime('%Y-%m-%d')
    end_dt   = datetime.datetime.utcfromtimestamp(p['usageEnd']   / 1000).strftime('%Y-%m-%d')
    print(f"── {example['label']} " + "─" * max(0, 52 - len(example['label'])))
    print(f"   Request : GET .../usage/aggregated/{example['group_by']}")
    print(f"   Params  : metricId={example['metric']}  transform={example['transform']}  "
          f"bucket={example['groupByHrs']}h  {start_dt} → {end_dt}")
    print(f"   Response: totalGroups={total_groups}  totalPeriodsCount={total_periods}")
    print(f"   Saved   : {out_path}")
    if periods:
        sample = {k: v for k, v in list(periods[0].items())[:5]}
        print(f"   Sample  : {json.dumps(sample)}")
    print()


Built 9 grouped example(s) (3 metric(s) × 3 dimensions):
  instances              groupByTenant      → transform=avg   label=last_30d_daily_avg_instances_by_tenant
  instances              groupByWorkspace   → transform=avg   label=last_30d_daily_avg_instances_by_workspace
  instances              groupByInstance    → transform=avg   label=last_30d_daily_avg_instances_by_instance
  users                  groupByTenant      → transform=avg   label=last_30d_daily_avg_users_by_tenant
  users                  groupByWorkspace   → transform=avg   label=last_30d_daily_avg_users_by_workspace
  users                  groupByInstance    → transform=avg   label=last_30d_daily_avg_users_by_instance
  api_calls              groupByTenant      → transform=sum   label=last_30d_daily_sum_api_calls_by_tenant
  api_calls              groupByWorkspace   → transform=sum   label=last_30d_daily_sum_api_calls_by_workspace
  api_calls              groupByInstance    → transform=sum   label=last_30d_daily_sum

/var/folders/mh/rhl3j_v5075fwz2dzshygytr0000gn/T/ipykernel_77376/9724077.py:74: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  start_dt = datetime.datetime.utcfromtimestamp(p['usageStart'] / 1000).strftime('%Y-%m-%d')
/var/folders/mh/rhl3j_v5075fwz2dzshygytr0000gn/T/ipykernel_77376/9724077.py:75: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  end_dt   = datetime.datetime.utcfromtimestamp(p['usageEnd']   / 1000).strftime('%Y-%m-%d')


── last_30d_daily_avg_instances_by_tenant ──────────────
   Request : GET .../usage/aggregated/groupByTenant
   Params  : metricId=instances  transform=avg  bucket=24h  2026-07-25 → 2026-08-24
   Response: totalGroups=4  totalPeriodsCount=50
   Saved   : data/grouped/apps.gori-agent-hub.cp.fyre.ibm.com/servicebrokercore/last_30d_daily_avg_instances_by_tenant.json
   Sample  : {"groupId": "7ef3d047-2729-402c-a956-3292c88329a7::", "groupTenantId": "7ef3d047-2729-402c-a956-3292c88329a7", "periodNumber": 18, "periodQuantity": 1, "periodStart": 1786406400000}

── last_30d_daily_avg_instances_by_workspace ───────────
   Request : GET .../usage/aggregated/groupByWorkspace
   Params  : metricId=instances  transform=avg  bucket=24h  2026-07-25 → 2026-08-24
   Response: totalGroups=6  totalPeriodsCount=74
   Saved   : data/grouped/apps.gori-agent-hub.cp.fyre.ibm.com/servicebrokercore/last_30d_daily_avg_instances_by_workspace.json
   Sample  : {"groupId": "7ef3d047-2729-402c-a956-3292c88329a7:202

## Next Steps

Your data has been fetched and stored in `data/` ✅

### Run the notebooks in order:

| # | Notebook | Reads from |
|---|---|---|
| 1 | **`Processing - Raw Usage Data.ipynb`** | `data/raw/` |
| 2 | **`Processing - Aggregated Usage Data.ipynb`** | `data/aggregated/` |
| 3 | **`Processing - Grouped Usage Data.ipynb`** | `data/grouped/` |
| 4 | **`Use Case - Telemetry.ipynb`** | **based on usecase |
| 5 | **`Use Case - Billing.ipynb`** | **based on usecase |

### Data folder layout

Files are written to `data/<type>/<APP_DOMAIN>/<SERVICE_ID>/` — one folder per service per environment.

```
data/
├── raw/          ← one record per (instance, metric, time-window)
├── aggregated/   ← time-series totals collapsed across all instances
└── grouped/      ← time-series totals broken down by tenant / workspace / instance
```

All three folders are safe to use offline — the downstream notebooks read from disk and do not call the API.